In [7]:
import pandas as pd

df_extracted = pd.read_csv("Test with manual labels.csv")
df_extracted = df_extracted[:60]
df_extracted

,ID,Land Surveyor,Surveyed For,Certified date,Total Area,Unit of Measurement,Address,Parish,LT Num
0,7703-078,Ronald R. Greene,Nero Millington,2020-09-14,430.8,sq m,Lot 636 Ruby Development,Davis,77.03.07.015
1,8606-095,Lennox J. Reid,Collymore Taylor,2021-10-14,428.7,sq m,Lot 7A Mangrove Park,Davis,86.06.07.090
2,7703-064,Ronald R. Greene,Frank Frank,2019-01-18,661.8,sq m,Lot 632 Ruby Development,Frank,77.03.08.034
3,7703-101,Michael L. Banfield,Smith Roachford,2022-08-27,527.3,sq m,"Lot 114 Ruby Park, Ruby Development",St. Philip,77.03.02.054
4,7707-114,Michael H Hutchinson,Liam Worrell,2013-02-07,540.4,sq m,Lot 224 Union Hall,St. Philip,77.07.06.028
5,8604-111,Andrew R. Bannister,Kerold Y. Davis,2013-10-14,1990.9,sq m,KIRTONS,SAINT PHILIP,86.04.02.011
6,7703-049,O'BRIEN ST.E WORRELL,Jones Emma,2016-06-03,602.5,sq m,"Lot 218, Ruby Development",St Philip,77.03.03.019
7,7706-060,Andre P.G. Clarke,Ann-Marie Warner,2016-12-28,4180.9,sq m,"Lot A, Union Road",St. Philip,77.06.03.016
8,7707-141,Lee B S Brathwaite,Henry V. Belgrave,2017-12-21,604.7,sq m,Lot 182 Union Hall,St. Philip,77.07.05.018
9,7707-152,Kenneth D. Ward,Jackson Jones,2018-07-27,580.2,sq m,"Lot 298, Union Hall Development",St. Philip,77.07.01.056


In [ ]:
import re
import numpy as np

def clean_land_surveyor_names(names_array):
    """
    Clean land surveyor names by removing middle initials while preserving:
    - Names that start with initials (like H.A. King)
    - Names with nicknames in quotes
    - Names with compound elements like St. Clair
    - Professional designations like JP
    """

    def clean_single_name(name):
        if pd.isna(name) or not name or name.strip() == '':
            return name

        name = str(name).strip()

        # Skip names that start with initials (like "H.A King" or "H.A. King")
        if re.match(r'^[A-Z]\.?\s*[A-Z]\.?\s+', name):
          return name

        # Skip names with quotes (nicknames like D.C "Vallan" Franklin JP)
        if '"' in name:
            return name

        # Skip single names (like "Simba")
        if len(name.split()) <= 1:
            return name

        # Protect "St." in compound names like "Michelle E. St. Clair"
        name_protected = name.replace(' St. ', ' PROTECTED_ST ')

        # Remove middle initials patterns:
        # 1. Single initials: "Lennox J Reid" to "Lennox Reid"
        name_cleaned = re.sub(r'\s+[A-Z]\.?\s+', ' ', name_protected)

        # 2. Multiple initials: "Jamal K.L. Gaskin" to "Jamal Gaskin"
        name_cleaned = re.sub(r'\s+[A-Z]\.[A-Z]\.?\s+', ' ', name_cleaned)

        # 3. Space-separated initials: "Lee B S Brathwaite" to "Lee Brathwaite"
        name_cleaned = re.sub(r'\s+[A-Z]\s+[A-Z]\s+', ' ', name_cleaned)

        # 4. Complex patterns like "Lee B.S Brathwaite" or "Sekani H.C Franklin"
        name_cleaned = re.sub(r'\s+[A-Z]\.[A-Z]\s+', ' ', name_cleaned)

        # 5. Handle remaining single initials that might be left
        name_cleaned = re.sub(r'\s+[A-Z]\.?\s+', ' ', name_cleaned)

        # Restore protected "St."
        name_cleaned = name_cleaned.replace(' PROTECTED_ST ', ' St. ')

        # Clean up multiple spaces and trim
        name_cleaned = re.sub(r'\s+', ' ', name_cleaned).strip()

        return name_cleaned

    # Apply cleaning to each name in the array
    if isinstance(names_array, np.ndarray):
        return np.array([clean_single_name(name) for name in names_array])
    elif isinstance(names_array, (list, pd.Series)):
        return [clean_single_name(name) for name in names_array]
    else:
        return clean_single_name(names_array)

# Create working copy
data_expanded = df_extracted.copy()

# Apply to both name columns
data_expanded['Land Surveyor'] = clean_land_surveyor_names(data_expanded['Land Surveyor'])
data_expanded['Surveyed For'] = clean_land_surveyor_names(data_expanded['Surveyed For'])

print("Names cleaned (middle initials removed)")

In [9]:
def clean_target_survey(text: str) -> str:
    """Lowercase, remove periods and commas, normalize spaces"""
    text = text.lower()
    text = re.sub(r"[.,]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
def format_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Create TargetSurvey and keep only required columns"""
    df["TargetSurvey"] = (
        df["Land Surveyor"].astype(str).str.strip() + " " +
        df["Surveyed For"].astype(str).str.strip() + " " +
        df["Address"].astype(str).str.strip()
    ).apply(clean_target_survey)

    columns_to_keep = [
        'ID', 'TargetSurvey', 'Certified date', 'Total Area',
        'Unit of Measurement', 'Parish', 'LT Num'
    ]
    return df[columns_to_keep]

final_data = format_dataset(data_expanded)

print("Final dataset created")
print(f"\nColumns: {final_data.columns.tolist()}")
print(f"\nTotal records: {len(final_data)}")
final_data.head(10)

In [11]:
# Save to CSV
final_data.to_csv('text_extraction_manual_labels_final.csv', index=False)

final_data

,ID,TargetSurvey,Certified date,Total Area,Unit of Measurement,Parish,LT Num
0,7703-078,ronald greene nero millington lot 636 ruby dev...,2020-09-14,430.8,sq m,Davis,77.03.07.015
1,8606-095,lennox reid collymore taylor lot 7a mangrove park,2021-10-14,428.7,sq m,Davis,86.06.07.090
2,7703-064,ronald greene frank frank lot 632 ruby develop...,2019-01-18,661.8,sq m,Frank,77.03.08.034
3,7703-101,michael banfield smith roachford lot 114 ruby ...,2022-08-27,527.3,sq m,St. Philip,77.03.02.054
4,7707-114,michael hutchinson liam worrell lot 224 union ...,2013-02-07,540.4,sq m,St. Philip,77.07.06.028
5,8604-111,andrew bannister kerold davis kirtons,2013-10-14,1990.9,sq m,SAINT PHILIP,86.04.02.011
6,7703-049,o'brien st e worrell jones emma lot 218 ruby d...,2016-06-03,602.5,sq m,St Philip,77.03.03.019
7,7706-060,andre clarke ann-marie warner lot a union road,2016-12-28,4180.9,sq m,St. Philip,77.06.03.016
8,7707-141,lee brathwaite henry belgrave lot 182 union hall,2017-12-21,604.7,sq m,St. Philip,77.07.05.018
9,7707-152,kenneth ward jackson jones lot 298 union hall ...,2018-07-27,580.2,sq m,St. Philip,77.07.01.056
